In [1]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from tabulate import tabulate
import numpy as np
import geopandas as gpd

In [2]:
train_panel_in = "../../../data/final/train_panel.parquet"
train_stations_in = "../../../data/interim/train_stations.geojson"

did_out = "../../../reports/replication/did_simple.md"

In [3]:
train_panel = pd.read_parquet(train_panel_in)
train_stations = gpd.read_file(train_stations_in)

# Preprocessing

In [4]:
dotw_names = {f"C(dotw)[T.{i}]":j for i,j in zip([1,2,3,4],['Tuesday','Wednesday','Thursday','Friday'])}

In [5]:
damen_mask = (train_stations['line'] == 'green') & (train_stations['station_name'] == 'Damen')
damen_id = train_stations[damen_mask]['map_id'].iloc[0]

In [6]:
# The DNC was only 4 days, not even friday. so weekends are uninformative.
# TODO: don't hard-code because we want to include more days for robustness check
dotw_mask = (0 <= train_panel['dotw']) & (train_panel['dotw'] <= 3)
# Since the Damen Green line stop opened the same week, it is unidentifiable
damen_mask = (train_panel['id'] != damen_id)
assert damen_mask.sum() < len(train_panel)
train_panel = train_panel[dotw_mask & damen_mask]

# DiffNDiff

In [7]:
def mask_colinear(df, groupby, coef):
    test = df.groupby(groupby)[coef].nunique() > 1
    mask = df[groupby].isin(test[test].index)
    if not all(test):
        print(f"Dropping {sum(~test)} {groupby}, {sum(~mask)} obs invariant to {coef}")
    return mask

def did_colinearity(df):
    mask = np.full_like(df.index, True)
    mask &= mask_colinear(df, 'id', 'DNC')
    mask &= mask_colinear(df, 'date', 'UCMP')
    df = df.loc[mask]
    return df

def model_transit_did(df):
    df = df.sort_values('date')
    unit_obs = df.groupby('id').size()
    df['weight'] = df['id'].map(unit_obs).apply(lambda x: 1/x)
    # Assert that spatial units have time variance (only need to test group inclusion, not every variable.)
    df = did_colinearity(df)

    formula = f"""logrides ~ UCMP : DNC + id + time"""
    did_model = sm.WLS.from_formula(formula, df, weights=df['weight']).fit(cov_type='cluster',cov_kwds={'groups': df['id']})
    return did_model

In [16]:
import plotly.express as px
plot_data = train_panel.copy()
df_transformed = (
    plot_data.groupby(['UCMP', 'time'], as_index=False)['logrides']
    .mean()
    .sort_values(['UCMP', 'time'])
)

df_transformed['percent_value'] = (
    df_transformed.groupby('UCMP')['logrides']
    .transform(lambda x: (x / x.mean()))
)
fig = px.line(df_transformed, x='time', y='logrides', color='UCMP')
fig.add_vline(plot_data[plot_data['DNC'] == 1]['time'].min())

In [8]:
train_did_model = model_transit_did(train_panel)
train_did_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:               logrides   R-squared:                       0.003
Model:                            WLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     35.82
Date:                Mon, 12 Oct 2026   Prob (F-statistic):           3.06e-17
Time:                        11:39:18   Log-Likelihood:                -10167.
No. Observations:                7896   AIC:                         2.034e+04
Df Residuals:                    7892   BIC:                         2.037e+04
Df Model:                           3                                         
Covariance Type:              cluster                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.9245      6.397      0.613      0.540      -8.614      16.463
UCMP:DNC      -0.0123      0.272     -0.045      0.964      -0.546       0.521
id          8.159e-05      0.000      0.520      0.603      -0.000       0.000
time           0.0008   8.78e-05      9.429      0.000       0.001       0.001
==============================================================================
Omnibus:                      299.329   Durbin-Watson:                   2.016
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              132.255
Skew:                          -0.026   Prob(JB):                     1.91e-29
Kurtosis:                       2.368   Cond. No.                     3.61e+06
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
[2] The condition number is large, 3.61e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [9]:
headers = ["DiD (Train)"]

did_summary = summary_col([train_did_model], 
            model_names=headers, 
            info_dict={"N":lambda x:(x.nobs)},
            stars=True,
            regressor_order=["UCMP:DNC", "id", "time"],
            drop_omitted=True)

did_summary.tables[0].index = (did_summary.tables[0].index
                               .str.replace('DNC','During DNC')
                               .str.replace('UCMP','Near DNC')
                               .str.replace('time','time trend')
                               .str.replace('id','unit fe'))

did_table = tabulate(did_summary.tables[0],
                   headers=[""] + headers,
                   showindex=True,
                   tablefmt='github')
print(did_table)
with open(did_out,"w") as f:
    f.write(did_table)

# !cp {did_out} ../../../../eric-mc2-cv/static/uploads/

|                     | DiD (Train)   |
|---------------------|---------------|
| Near DNC:During DNC | -0.0123       |
|                     | (0.2723)      |
| unit fe             | 0.0001        |
|                     | (0.0002)      |
| time trend          | 0.0008***     |
|                     | (0.0001)      |
| R-squared           | 0.0025        |
| R-squared Adj.      | 0.0022        |
| N                   | 7896.0        |


## DiD Impact

In [10]:
beta3 = did_summary.tables[0].xs('Near DNC:During DNC').str.replace('*','').astype(float)
main_pct = np.exp(beta3) - 1
pct3 = main_pct.apply("{:.1%}".format)
pct3

DiD (Train)    -1.2%
Name: Near DNC:During DNC, dtype: object